1. Introduction

In this notebook, I followed the PyTorch tutorials for Learn the Basics, Quickstart, and Tensors.
The goal was to understand how PyTorch works for building and training simple machine learning models.

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

PyTorch can run on GPU if available, otherwise CPU. torch, nn, and DataLoader are core parts of PyTorch. torchvision helps with datasets like images.

In [ ]:
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor()
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor()
)

train_dataloader = DataLoader(training_data, batch_size=64)
test_dataloader = DataLoader(test_data, batch_size=64)

Datasets can be downloaded easily using torchvision.
DataLoader helps load data in batches instead of all at once.
Batching is important for performance and training efficiency.

In [ ]:
for X, y in test_dataloader:
    print("Shape of X:", X.shape)
    print("Shape of y:", y.shape)
    break

Each batch contains multiple images and labels.
Data has shape [batch_size, channels, height, width].
Labels are just numbers representing classes.

In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.layers = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        return self.layers(x)

model = NeuralNetwork().to(device)
print(model)

Models are created by extending nn.Module.
Flatten converts image data into a vector.
Layers like Linear and ReLU are used to build networks.
The forward function defines how data flows through the model.

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

Loss function measures how wrong the model is.
CrossEntropyLoss is commonly used for classification.
Optimizer updates the model weights during training.

In [ ]:
def train(dataloader, model, loss_fn, optimizer):
    model.train()
    for X, y in dataloader:
        X, y = X.to(device), y.to(device)

        pred = model(X)
        loss = loss_fn(pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()